# SEC Financial Data Explorer

This notebook demonstrates how actBI integrates **SEC EDGAR financial data** to power business intelligence.

**What's available:**
- 92M+ financial data points from SEC filings
- 7,700+ public companies
- 12 financial themes (profitability, revenue, debt, etc.)
- Segment data (geographic, product, business breakdowns)
- Data from 10-K, 10-Q, and 8-K filings

We'll explore:
1. **Discovery** - What data is available?
2. **Deep Dive** - Apple's financial health
3. **Comparison** - Tech giants (AAPL vs MSFT)
4. **Consumer Brands** - Starbucks & McDonald's
5. **Segment Analysis** - Where does revenue come from?
6. **Scale** - Market-wide analysis

## Setup

In [1]:
import os

import pandas as pd
import plotly.express as px

# Point to pipelines data directory
os.environ["ACTBI_DATA_PATH"] = "../pipelines/_data/assets"

from shared import data

# Use local environment (materialized parquet files)
data.use_env("local")
print(f"Environment: {data.current_env()}")

Environment: local


## 1. Discovery: What Data is Available?

The `data.sec` module provides easy access to SEC financial data with theme-based filtering.

In [2]:
# How many companies do we have?
tickers = data.sec.list_tickers()
print(f"Total companies available: {len(tickers):,}")
print(f"\nSample tickers: {tickers[:20]}")

Total companies available: 7,753

Sample tickers: ['A', 'AA', 'AAAU', 'AACB', 'AACBR', 'AACBU', 'AACG', 'AAL', 'AAM', 'AAM-UN', 'AAM-WT', 'AAME', 'AAMI', 'AAOI', 'AAON', 'AAP', 'AAPI', 'AAPL', 'AAQL', 'AARD']


In [3]:
# What financial themes can we query?
from shared.data.themes import FINANCIAL_THEMES

print("Available financial themes:\n")
for name, theme in FINANCIAL_THEMES.items():
    print(f"  {name}: {theme.description}")

Available financial themes:

  profitability: Net income and operating performance metrics
  revenue: Sales and revenue recognition
  balance_sheet: Assets, liabilities, and equity totals
  cash_position: Cash and cash equivalents on hand
  cash_flow_operations: Operating cash flows and working capital adjustments
  cash_flow_investing: CapEx, acquisitions, and asset purchases/sales
  cash_flow_financing: Debt/equity issuance, dividends, and buybacks
  earnings_per_share: EPS and shares outstanding
  debt_leverage: Debt levels and interest costs
  shareholder_returns: Dividends and stock buybacks
  expenses: Operating costs, R&D, and SG&A
  taxes: Tax expense and effective rates


In [4]:
# What XBRL concepts are in the 'profitability' theme?
concepts = data.sec.list_concepts("profitability")
print(f"Profitability concepts ({len(concepts)}):")
for c in concepts:
    print(f"  - {c}")

Profitability concepts (8):
  - GrossProfit
  - IncomeLossFromContinuingOperations
  - IncomeLossFromContinuingOperationsBeforeIncomeTaxesExtraordinaryItemsNoncontrollingInterest
  - NetIncomeLoss
  - NetIncomeLossAvailableToCommonStockholdersBasic
  - NetIncomeLossAvailableToCommonStockholdersDiluted
  - OperatingIncomeLoss
  - ProfitLoss


## 2. Deep Dive: Apple's Financial Health

Let's analyze Apple's profitability over the past 6 years.

In [ ]:
# Load Apple's profitability data (annual filings only)
apple_profit = data.sec.financials(
    ticker="AAPL",
    theme="profitability",
    fiscal_period="FY",
    long=True,
)

# SEC filings contain both quarterly and annual totals.
# Use max per fiscal year to get annual totals (always larger than quarterly)
key_metrics = ["NetIncomeLoss", "OperatingIncomeLoss", "GrossProfit"]
apple_annual = (
    apple_profit
    .query("concept in @key_metrics")
    .assign(fiscal_year=lambda x: pd.to_datetime(x["end_date"]).dt.year)
    .query("fiscal_year >= 2019 and fiscal_year <= 2024")
    .groupby(["fiscal_year", "concept"])
    .agg({"value": "max", "label": "first"})
    .reset_index()
)

print("Apple profitability metrics (2019-2024, in billions):")
(
    apple_annual
    .pivot(index="fiscal_year", columns="concept", values="value")
    .apply(lambda x: x / 1e9)
    .round(1)
)

In [ ]:
# Visualize Apple's profitability trend
fig = px.line(
    apple_annual,
    x="fiscal_year",
    y="value",
    color="concept",
    title="Apple Profitability Metrics (2019-2024)",
    labels={
        "value": "Amount",
        "fiscal_year": "Fiscal Year",
        "concept": "Metric",
    },
    markers=True,
)

# Format y-axis as billions
fig.update_layout(
    yaxis=dict(
        tickformat="$,.0f",
        tickvals=[50e9, 100e9, 150e9, 200e9],
        ticktext=["$50B", "$100B", "$150B", "$200B"],
    ),
    xaxis=dict(dtick=1),
)
fig.update_traces(hovertemplate="FY%{x}: $%{y:,.0f}<extra></extra>")
fig.show()

### Apple Financial Summary

Apple generated **$94B in net income** in FY2024 with gross profit reaching **$181B**. Despite slight declines from the 2022 peak, Apple maintains exceptional profitability with operating margins consistently above 30%.

## 3. Tech Giants: Apple vs Microsoft

How do the two largest tech companies compare on profitability?

In [ ]:
# Load profitability data for both companies
tech_profit = data.sec.financials(
    ticker=["AAPL", "MSFT"],
    theme="profitability",
    fiscal_period="FY",
    long=True,
)

# Use max value per year for annual totals
tech_annual = (
    tech_profit
    .query('concept in ["NetIncomeLoss", "OperatingIncomeLoss"]')
    .assign(fiscal_year=lambda x: pd.to_datetime(x["end_date"]).dt.year)
    .query("fiscal_year >= 2022 and fiscal_year <= 2024")
    .groupby(["ticker", "fiscal_year", "concept"])
    .agg({"value": "max"})
    .reset_index()
)

# Latest year comparison
latest = tech_annual.query("fiscal_year == 2024")
print("FY2024 Profitability Comparison (in billions):\n")
(
    latest
    .pivot(index="ticker", columns="concept", values="value")
    .apply(lambda x: x / 1e9)
    .round(1)
)

In [8]:
# Side-by-side comparison chart
fig = px.bar(
    tech_annual.query('concept == "NetIncomeLoss"'),
    x="fiscal_year",
    y="value",
    color="ticker",
    barmode="group",
    title="Net Income: Apple vs Microsoft (2022-2024)",
    labels={"value": "Net Income", "fiscal_year": "Fiscal Year"},
)

fig.update_layout(
    yaxis=dict(
        tickvals=[0, 25e9, 50e9, 75e9, 100e9],
        ticktext=["$0", "$25B", "$50B", "$75B", "$100B"],
    ),
    xaxis=dict(dtick=1),
)
fig.update_traces(hovertemplate="%{x}: $%{y:,.0f}<extra></extra>")
fig.show()

### Tech Giants Summary

Apple leads in absolute net income at **$94B** vs Microsoft's **$88B** in FY2024. Both companies show strong operating income, with Apple at **$123B** and Microsoft at **$109B**.

## 4. Consumer Brands: Starbucks & McDonald's

actBI started with a focus on coffee companies. Let's compare Starbucks with McDonald's across multiple financial dimensions.

In [ ]:
# Load revenue data for consumer brands
consumer_rev = data.sec.financials(
    ticker=["SBUX", "MCD"],
    theme="revenue",
    fiscal_period="FY",
    long=True,
)

# Use max value per year for annual totals
revenue_concepts = [
    "Revenues",
    "RevenueFromContractWithCustomerExcludingAssessedTax",
]
rev_annual = (
    consumer_rev
    .query("concept in @revenue_concepts")
    .assign(fiscal_year=lambda x: pd.to_datetime(x["end_date"]).dt.year)
    .query("fiscal_year >= 2019 and fiscal_year <= 2024")
    .groupby(["ticker", "fiscal_year"])
    .agg({"value": "max"})
    .reset_index()
)

print("Annual Revenue (in billions):")
(
    rev_annual
    .pivot(index="fiscal_year", columns="ticker", values="value")
    .apply(lambda x: x / 1e9)
    .round(1)
)

In [10]:
# Revenue trend comparison
fig = px.line(
    rev_annual,
    x="fiscal_year",
    y="value",
    color="ticker",
    title="Revenue: Starbucks vs McDonald's (2019-2024)",
    labels={"value": "Revenue", "fiscal_year": "Fiscal Year"},
    markers=True,
)

fig.update_layout(
    yaxis=dict(
        tickvals=[0, 10e9, 20e9, 30e9, 40e9],
        ticktext=["$0", "$10B", "$20B", "$30B", "$40B"],
    ),
    xaxis=dict(dtick=1),
)
fig.update_traces(hovertemplate="FY%{x}: $%{y:,.0f}<extra></extra>")
fig.show()

In [ ]:
# Compare shareholder returns (dividends + buybacks)
consumer_returns = data.sec.financials(
    ticker=["SBUX", "MCD"],
    theme="shareholder_returns",
    fiscal_period="FY",
    long=True,
)

returns_concepts = [
    "PaymentsOfDividendsCommonStock",
    "PaymentsForRepurchaseOfCommonStock",
]
returns_annual = (
    consumer_returns
    .query("concept in @returns_concepts")
    .assign(fiscal_year=lambda x: pd.to_datetime(x["end_date"]).dt.year)
    .query("fiscal_year >= 2022 and fiscal_year <= 2024")
    .groupby(["ticker", "fiscal_year", "concept"])
    .agg({"value": "max"})
    .reset_index()
)

print("Shareholder Returns (2024, in billions):")
returns_annual.query("fiscal_year == 2024").pivot(
    index="ticker", columns="concept", values="value"
).apply(lambda x: x / 1e9).round(2)

### Consumer Brands Summary

Starbucks generates higher revenue (**$36B**) than McDonald's (**$26B**) in FY2024. Both companies recovered strongly post-pandemic and continue to grow, with Starbucks showing 36% revenue growth since 2019.

## 5. Segment Analysis: Where Does the Revenue Come From?

Beyond total revenue, SEC filings contain **segment data** showing how companies break down their business by geography, product line, and operating segment. This reveals strategic priorities and market exposure.

**Canonical Segment Names:** Segment names are normalized to snake_case (e.g., `us`, `china`, `intl_operated`) for consistency across years. Different raw SEC names like `US`, `UnitedStates`, and `USMarket` all map to `us`.

In [ ]:
# What segments does Starbucks report?
# Check each segment type
segment_types = ["geographic", "product", "business"]

print("Starbucks segment data by type:\n")
for seg_type in segment_types:
    df = data.sec.segments(ticker="SBUX", segment_type=seg_type)
    if len(df) > 0:
        unique_segments = df["segment_name"].unique().tolist()
        suffix = "..." if len(unique_segments) > 8 else ""
        print(f"  {seg_type}: {unique_segments[:8]}{suffix}")
    else:
        print(f"  {seg_type}: (no data)")

In [13]:
# Starbucks Geographic Revenue Over Time
sbux_geo = data.sec.segments(
    ticker="SBUX",
    segment_type="geographic",
    concept="Revenues",
)

# Clean and aggregate by year
sbux_geo_annual = (
    sbux_geo
    .dropna(subset=["fiscal_year", "value"])
    .assign(fiscal_year=lambda x: x["fiscal_year"].astype(int))
    .query("fiscal_year >= 2019 and fiscal_year <= 2024")
    .groupby(["fiscal_year", "segment_name"])
    .agg({"value": "max"})
    .reset_index()
    .assign(value_billions=lambda x: x["value"] / 1e9)
)

# Pivot for display
geo_pivot = sbux_geo_annual.pivot(
    index="fiscal_year", columns="segment_name", values="value_billions"
).round(1)

print("Starbucks Revenue by Geography (in billions):\n")
geo_pivot

Starbucks Revenue by Geography (in billions):



segment_name,china,international,us
fiscal_year,,,
2019,NaN,7.9,18.6
2020,2.9,5.0,18.6
2021,3.7,5.0,20.4
2022,3.7,5.9,23.4
2023,3.7,6.5,26.4
2024,3.1,6.5,26.7


In [ ]:
# Visualize Starbucks Geographic Revenue Evolution
fig = px.area(
    sbux_geo_annual,
    x="fiscal_year",
    y="value_billions",
    color="segment_name",
    title="Starbucks Revenue by Geography (2019-2024)",
    labels={
        "value_billions": "Revenue (Billions)",
        "fiscal_year": "Fiscal Year",
        "segment_name": "Region",
    },
)

fig.update_layout(
    yaxis=dict(tickprefix="$", ticksuffix="B"),
    xaxis=dict(dtick=1),
    legend=dict(title="Region"),
)
fig.show()

In [ ]:
# McDonald's Business Segments - Different reporting structure
print("McDonald's segment data by type:\n")
for seg_type in ["geographic", "product", "business"]:
    df = data.sec.segments(ticker="MCD", segment_type=seg_type)
    if len(df) > 0:
        unique_segments = df["segment_name"].unique().tolist()
        suffix = "..." if len(unique_segments) > 8 else ""
        print(f"  {seg_type}: {unique_segments[:8]}{suffix}")
    else:
        print(f"  {seg_type}: (no data)")

In [ ]:
# McDonald's Business Segment Revenue Over Time
mcd_biz = data.sec.segments(
    ticker="MCD",
    segment_type="business",
    concept="Revenues",
)

# Clean and aggregate by year - filter to main operating segments
mcd_biz_annual = (
    mcd_biz
    .dropna(subset=["fiscal_year", "value"])
    .assign(fiscal_year=lambda x: x["fiscal_year"].astype(int))
    .query("fiscal_year >= 2019 and fiscal_year <= 2024")
    # Filter to segments with significant revenue (>$1B)
    .query("value > 1e9")
    .groupby(["fiscal_year", "segment_name"])
    .agg({"value": "max"})
    .reset_index()
    .assign(value_billions=lambda x: x["value"] / 1e9)
)

# Keep only segments that appear in multiple years
segment_counts = mcd_biz_annual.groupby("segment_name").size()
main_segments = segment_counts[segment_counts >= 3].index
mcd_biz_annual = mcd_biz_annual[mcd_biz_annual["segment_name"].isin(main_segments)]

print("McDonald's Revenue by Operating Segment (in billions):\n")
mcd_biz_annual.pivot(
    index="fiscal_year", columns="segment_name", values="value_billions"
).round(1)

In [ ]:
# Visualize McDonald's Segment Evolution
fig = px.bar(
    mcd_biz_annual,
    x="fiscal_year",
    y="value_billions",
    color="segment_name",
    title="McDonald's Revenue by Operating Segment (2019-2024)",
    labels={
        "value_billions": "Revenue (Billions)",
        "fiscal_year": "Fiscal Year",
        "segment_name": "Segment",
    },
    barmode="group",
)

fig.update_layout(
    yaxis=dict(tickprefix="$", ticksuffix="B"),
    xaxis=dict(dtick=1),
    legend=dict(title="Segment"),
)
fig.show()

### Segment Analysis Summary

**Starbucks** reveals a heavily US-centric business:
- **US market dominates** at ~$27B (75% of revenue)
- **China represents ~$3B** (~8%) - a strategic growth market
- **International ex-China** contributes the remainder

**McDonald's** uses a different reporting structure:
- **International Operated Markets (`intl_operated`)** lead revenue (company-owned stores abroad)
- **US Market (`us`)** is consistent across all reporting years
- **International Licensed (`intl_licensed`)** represents franchise markets

The segment data shows how these companies approach global expansion differently:
- Starbucks is **company-operated focused** with heavy US concentration
- McDonald's is **franchise-heavy** with more geographic diversification

**Note:** Segment names are normalized to canonical snake_case names (e.g., `us`, `intl_operated`) for consistency across years. Use `use_raw_names=True` to access original SEC names.

## 6. Market-Wide Analysis

The real power is analyzing trends across thousands of companies.

In [18]:
# Find top companies by revenue (USD-denominated only)
all_revenue = data.sec.financials(
    theme="revenue",
    fiscal_period="FY",
    unit="USD",  # Filter to USD to exclude foreign currency filings
    long=True,
)

# Get max revenue per company for 2024
top_revenue = (
    all_revenue
    .query('concept == "Revenues"')
    .assign(fiscal_year=lambda x: pd.to_datetime(x["end_date"]).dt.year)
    .query("fiscal_year == 2024")
    .groupby(["ticker", "company_name"])
    .agg({"value": "max"})
    .reset_index()
    .sort_values("value", ascending=False)
    .head(15)
)

top_revenue["revenue_billions"] = (top_revenue["value"] / 1e9).round(1)
print("Top 15 Companies by Revenue (FY2024, USD):\n")
top_revenue[["ticker", "company_name", "revenue_billions"]]

Top 15 Companies by Revenue (FY2024, USD):



,ticker,company_name,revenue_billions
2528,WMT,Walmart Inc.,648.1
2382,UNH,UNITEDHEALTH GROUP INC,400.3
625,CVS,CVS HEALTH Corp,372.8
361,BRK-B,BERKSHIRE HATHAWAY INC,371.4
360,BRK-A,BERKSHIRE HATHAWAY INC,371.4
2571,XOM,EXXON MOBIL CORP,349.6
560,COR,"Cencora, Inc.",294.0
567,COST,COSTCO WHOLESALE CORP /NEW,254.5
478,CI,Cigna Group,247.1
405,CAH,CARDINAL HEALTH INC,226.8


In [ ]:
# Visualize top companies
fig = px.bar(
    top_revenue.head(10),
    x="ticker",
    y="value",
    title="Top 10 Companies by Revenue (FY2024)",
    labels={"value": "Revenue", "ticker": "Company"},
    hover_data=["company_name"],
)

fig.update_layout(
    yaxis=dict(
        tickvals=[0, 100e9, 200e9, 300e9, 400e9, 500e9, 600e9],
        ticktext=["$0", "$100B", "$200B", "$300B", "$400B", "$500B", "$600B"],
    ),
)
hover_tpl = "%{customdata[0]}<br>Revenue: $%{y:,.0f}<extra></extra>"
fig.update_traces(hovertemplate=hover_tpl)
fig.show()

In [20]:
# Quick stats about the dataset
tickers = data.sec.list_tickers()
print("Dataset Statistics:")
print(f"  Total companies with SEC data: {len(tickers):,}")
print("  Financial themes available: 12")
print("  Data sources: 10-K, 10-Q, 8-K filings")

Dataset Statistics:
  Total companies with SEC data: 7,753
  Financial themes available: 12
  Data sources: 10-K, 10-Q, 8-K filings


## Conclusion

This notebook demonstrated actBI's SEC financial data integration:

- **7,700+ companies** with standardized XBRL financial data
- **Theme-based queries** map business questions to technical XBRL concepts
- **Segment analysis** reveals geographic, product, and business breakdowns
- **Easy comparison** across companies and time periods
- **Interactive visualizations** for exploring financial trends

This data powers actBI's AI-driven business intelligence, enabling natural language queries like:
- *"Is Apple more profitable than Microsoft?"*
- *"How has Starbucks' revenue grown?"*
- *"What percentage of Starbucks revenue comes from China?"*
- *"Which companies have the highest shareholder returns?"*